## 1. DuckDB 연결 설정

* **하는 일**: DuckDB 파일(`mimic_total.duckdb`)에 연결함.
* **변수 의미**

  * `db_path`: DuckDB 파일 경로임.
  * `con`: DuckDB 커넥션 객체임. 이후 모든 SQL 실행 핸들임.

In [1]:
import duckdb

db_path = '../data/duckdb/mimic_total.duckdb'
con = duckdb.connect(db_path)

## 2. 슬라이딩 윈도우 CSV 경로 지정

* **하는 일**: 1시간 단위로 이미 슬라이딩된 코호트 CSV 경로를 지정함.
* **변수 의미**

  * `window_csv_path`: `cohort_sliding_window_v2.csv` 파일 경로 문자열임.

In [2]:
window_csv_path = '../data/processed/cohort_sliding_window_v2.csv'

## 3. CSV 스키마 확인(DESCRIBE)

* **하는 일**: `read_csv_auto`로 CSV를 읽어 컬럼/타입을 확인함.

In [3]:
con.execute(f"""
DESCRIBE SELECT * FROM read_csv_auto('{window_csv_path}');
""").df()


,column_name,column_type,null,key,default,extra
0,subject_id,BIGINT,YES,None,None,None
1,hadm_id,BIGINT,YES,None,None,None
2,stay_id,BIGINT,YES,None,None,None
3,intime,TIMESTAMP,YES,None,None,None
4,outtime,TIMESTAMP,YES,None,None,None
5,los,DOUBLE,YES,None,None,None
6,first_careunit,VARCHAR,YES,None,None,None
7,last_careunit,VARCHAR,YES,None,None,None
8,anchor_age,BIGINT,YES,None,None,None
9,gender,VARCHAR,YES,None,None,None


* **변수 의미**

  * 반환 DF: CSV에 어떤 컬럼이 있는지(예: `observation_start_time`, `observation_end_time`, `stay_id`) 확인용임.
* **통계적 의미**

  * 분석 단위(윈도우) 정의에 필요한 키 컬럼이 존재하는지 검증함.
  * 타입이 TIMESTAMP로 잡히는지 확인해 이후 조인의 일치성(키 매칭 실패 방지)에 중요함.

## 4. `windows_1h` 뷰 생성(윈도우 테이블 정규화)

* **하는 일**

  * CSV에서 필요한 컬럼만 뽑아 `windows_1h` 뷰를 만듦.
  * `observation_start_time/end_time`를 `window_start/window_end`로 표준화함.

In [4]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW windows_1h AS
SELECT
  t.stay_id,
  t.observation_start_time AS window_start,
  t.observation_end_time   AS window_end
FROM read_csv_auto('{window_csv_path}') AS t
WHERE t.observation_start_time IS NOT NULL
  AND t.observation_end_time IS NOT NULL;
""")

con.execute("DESCRIBE windows_1h").df(), con.execute("SELECT * FROM windows_1h LIMIT 5").df()


(    column_name column_type null   key default extra
 0       stay_id      BIGINT  YES  None    None  None
 1  window_start   TIMESTAMP  YES  None    None  None
 2    window_end   TIMESTAMP  YES  None    None  None,
     stay_id        window_start          window_end
 0  37262027 2177-12-05 21:15:00 2177-12-06 03:15:00
 1  33653109 2187-07-28 16:47:19 2187-07-28 22:47:19
 2  38791957 2129-12-20 13:48:00 2129-12-20 19:48:00
 3  36469520 2111-11-14 02:06:00 2111-11-14 08:06:00
 4  36149581 2174-05-19 08:02:00 2174-05-19 14:02:00)

* **변수 의미**

  * `windows_1h`: (stay_id, window_start, window_end) 형태의 “1시간 분석 단위” 테이블임.
* **통계적 의미**

  * 이후 모든 결측률/ffill 영향/분포 계산은 **분모가 `windows_1h`의 전체 윈도우 수**로 정의됨.
  * 이 단계에서 분석 단위(샘플 단위)가 고정됨.

## 5. `gcs_events_all` 뷰 생성(GCS 원본 이벤트 구성)

* **하는 일**

  * `d_items`에서 GCS 3개 컴포넌트 itemid를 `label`로 식별함.
  * `chartevents`에서 해당 itemid의 측정값(`valuenum`)과 `charttime`을 추출함.
  * `(stay_id, charttime)` 단위로 `gcs_eye/gcs_verbal/gcs_motor`를 피벗 형태로 구성함.

In [5]:
con.execute("""
CREATE OR REPLACE TEMP VIEW gcs_events_all AS
WITH gcs_items AS (
  SELECT
    itemid,
    CASE
      WHEN label = 'GCS - Eye Opening' THEN 'eye'
      WHEN label = 'GCS - Verbal Response' THEN 'verbal'
      WHEN label = 'GCS - Motor Response' THEN 'motor'
    END AS part
  FROM d_items
  WHERE label IN ('GCS - Eye Opening','GCS - Verbal Response','GCS - Motor Response')
),
raw AS (
  SELECT
    ce.stay_id,
    TRY_CAST(ce.charttime AS TIMESTAMP) AS charttime,
    gi.part,
    TRY_CAST(ce.valuenum AS DOUBLE) AS val
  FROM chartevents ce
  JOIN gcs_items gi ON ce.itemid = gi.itemid
  WHERE TRY_CAST(ce.charttime AS TIMESTAMP) IS NOT NULL
)
SELECT
  stay_id,
  charttime,
  MAX(CASE WHEN part='eye' THEN val END)    AS gcs_eye,
  MAX(CASE WHEN part='verbal' THEN val END) AS gcs_verbal,
  MAX(CASE WHEN part='motor' THEN val END)  AS gcs_motor
FROM raw
GROUP BY stay_id, charttime;
""")


* **변수 의미**

  * `gcs_events_all`: GCS 이벤트 원자료를 “시간축 이벤트 테이블” 형태로 만든 뷰임.
* **통계적 의미**

  * MIMIC은 동일 시점에 E/V/M이 동시에 기록되지 않을 수 있음(비동기 기록 가능성 존재함).
  * 따라서 “행(row) 단위 결측”이 아니라 “시간 정렬 후 윈도우 집계 시 결측” 관점이 필요함.
  * 이 뷰는 이후 윈도우 집계의 모수(population)가 되는 이벤트 집합임.

## 6. `missing_compare`: ffill 없음 vs ffill60 결측률 비교

* **하는 일**

  * 각 1시간 윈도우에서 `gcs_motor`의 **윈도우 내 마지막 값(last in window)** 을 계산함.
  * 윈도우에 값이 없으면 `window_start` 직전 60분 내 마지막 값을 찾아 `ffill60` 후보로 사용함.
  * `motor_last_no_ffill` vs `motor_last_ffill60` 결측률을 비교함.

In [6]:
missing_compare = con.execute("""
WITH in_window AS (
  SELECT
    w.stay_id,
    w.window_start,
    w.window_end,
    arg_max(g.gcs_motor, g.charttime) AS motor_last_in_window,
    MAX(CASE WHEN g.gcs_motor IS NOT NULL THEN 1 ELSE 0 END) AS observed_flag
  FROM windows_1h w
  LEFT JOIN gcs_events_all g
    ON g.stay_id = w.stay_id
   AND g.charttime >= w.window_start
   AND g.charttime <  w.window_end
  GROUP BY 1,2,3
),
ffill60 AS (
  SELECT
    iw.*,
    (
      SELECT g2.gcs_motor
      FROM gcs_events_all g2
      WHERE g2.stay_id = iw.stay_id
        AND g2.gcs_motor IS NOT NULL
        AND g2.charttime < iw.window_start
        AND g2.charttime >= iw.window_start - INTERVAL '60 minutes'
      ORDER BY g2.charttime DESC
      LIMIT 1
    ) AS motor_last_prev60
  FROM in_window iw
),
final AS (
  SELECT
    motor_last_in_window AS motor_last_no_ffill,
    COALESCE(motor_last_in_window, motor_last_prev60) AS motor_last_ffill60
  FROM ffill60
)
SELECT
  COUNT(*) AS n_windows,
  SUM(CASE WHEN motor_last_no_ffill IS NULL THEN 1 ELSE 0 END) AS missing_no_ffill,
  ROUND(100.0 * SUM(CASE WHEN motor_last_no_ffill IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_rate_no_ffill_pct,
  SUM(CASE WHEN motor_last_ffill60 IS NULL THEN 1 ELSE 0 END) AS missing_ffill60,
  ROUND(100.0 * SUM(CASE WHEN motor_last_ffill60 IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS missing_rate_ffill60_pct
FROM final;
""").df()

missing_compare


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_windows,missing_no_ffill,missing_rate_no_ffill_pct,missing_ffill60,missing_rate_ffill60_pct
0,934312,82907.0,8.87,62748.0,6.72


* **변수 의미**

  * `missing_compare`: 전체 윈도우 수, 결측 개수/결측률(%)를 요약한 DF임.
  * 핵심 필드(쿼리 내부 개념)

    * `motor_last_in_window`: 윈도우 내부 마지막 관측값임.
    * `motor_last_prev60`: 직전 60분 내 마지막 관측값(보정 후보)임.
    * `motor_last_ffill60`: `COALESCE(in_window, prev60)` 결과임.
* **통계적 의미**

  * `missing_rate_no_ffill_pct`: “엄격한 관측 기반” 결측률임.
  * `missing_rate_ffill60_pct`: “제한적 보정(ffill60) 적용 후” 결측률임.
  * **“1시간 window에서 ffill로 결측률이 얼마나 줄었는지”** 를 수치로 보여줌
  * 두 값의 차이는 **ffill60이 추가로 ‘살린 윈도우’ 비율**을 의미함(데이터 커버리지 향상 정도를 정량화함).

* **ffill(≤60분)이 결측을 얼마나 줄였나?**
    * 전체 윈도우: 934,312
        * ffill 없음 결측: 82,907 (8.87%)
        * ffill60 결측: 62,748 (6.72%)

    * 핵심 정리
        * 절대 감소(퍼센트포인트): 8.87% → 6.72% = 2.15%p 감소
        * 상대 감소(결측 수 기준): 82,907 → 62,748
        * 감소량 = 20,159개
        * 상대 감소율 = 20,159 / 82,907 ≈ 24.3% 감소

    => ✅ 결론: ffill60은 결측을 “조금 줄이는 수준”이 아니라, 결측의 약 1/4을 제거

## 7. `observed_pattern`: 관측 신호(observed) vs ffill로 채움의 관계

* **하는 일**

  * 윈도우 내부에 관측이 있었는지(`observed_flag`)를 계산함.
  * 관측이 없더라도 ffill60으로 값이 생기면 `observed_or_ffilled_flag=1`로 표시함.
  * (observed_flag, observed_or_ffilled_flag) 교차표를 생성함.

In [ ]:
observed_pattern = con.execute("""
WITH in_window AS (
  SELECT
    w.stay_id,
    w.window_start,
    w.window_end,
    arg_max(g.gcs_motor, g.charttime) AS motor_last_in_window,
    MAX(CASE WHEN g.gcs_motor IS NOT NULL THEN 1 ELSE 0 END) AS observed_flag
  FROM windows_1h w
  LEFT JOIN gcs_events_all g
    ON g.stay_id = w.stay_id
   AND g.charttime >= w.window_start
   AND g.charttime <  w.window_end
  GROUP BY 1,2,3
),
ffill60 AS (
  SELECT
    iw.*,
    (
      SELECT g2.gcs_motor
      FROM gcs_events_all g2
      WHERE g2.stay_id = iw.stay_id
        AND g2.gcs_motor IS NOT NULL
        AND g2.charttime < iw.window_start
        AND g2.charttime >= iw.window_start - INTERVAL '60 minutes'
      ORDER BY g2.charttime DESC
      LIMIT 1
    ) AS motor_last_prev60
  FROM in_window iw
),
final AS (
  SELECT
    observed_flag,
    CASE
      WHEN motor_last_in_window IS NOT NULL THEN 1
      WHEN motor_last_prev60 IS NOT NULL THEN 1
      ELSE 0
    END AS observed_or_ffilled_flag
  FROM ffill60
)
SELECT
  observed_flag,
  observed_or_ffilled_flag,
  COUNT(*) AS n_windows,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM final
GROUP BY 1,2
ORDER BY 1,2;
""").df()

observed_pattern

,observed_flag,observed_or_ffilled_flag,n_windows,pct
0,0,0,62748,6.72
1,0,1,20159,2.16
2,1,1,851405,91.13


* **변수 의미**

  * `observed_pattern`: 두 플래그 조합별 윈도우 수/비율 DF임.
  * `observed_flag`: 윈도우 내부에 해당 컴포넌트 관측 이벤트가 1개라도 있으면 1임.
  * `observed_or_ffilled_flag`: 관측이 있거나 ffill로 값이 채워지면 1임.
* **통계적 의미**

  * `observed_flag`는 단순 결측이 아니라 **관측 프로세스(측정 행위)의 발생 여부**를 나타내는 변수임.
  * `(0,1)` 케이스(관측 없음 → ffill로만 값 존재)는 “관측 신호를 ffill이 얼마나 덮는지”를 의미함.
  * 즉, ffill 적용 시에도 관측 여부를 별도 피처로 유지해야 하는 근거를 제공함.

* 표가 주는 인사이트
    * 91.13%: 윈도우 안에 실제로 gcs_motor가 찍힘 (ffill 여부와 무관)
    * 2.16%: 윈도우 안엔 없었는데, 직전 60분 안에서 찾아서 ffill로 채운 케이스
    * 6.72%: 윈도우 안에도 없고, 직전 60분에도 없어서 끝까지 결측

    => ✅ 결론: ffill60이 “관측 과정 신호를 크게 덮어버린다”기보단, “윈도우 경계 근처에서 살짝 어긋나 찍힌 이벤트를 구제”하는 성격이 강함

## 8. 1시간 step 검증(윈도우 간격 확인)

* **하는 일**

  * 같은 stay_id 내에서 `window_start` 간 차이를 계산해 중앙값(분)을 확인함.

In [8]:
con.execute("""
WITH x AS (
  SELECT
    stay_id,
    window_start,
    LAG(window_start) OVER (PARTITION BY stay_id ORDER BY window_start) AS prev_start
  FROM windows_1h
)
SELECT
  approx_quantile(date_diff('minute', prev_start, window_start), [0.5]) AS median_step_minutes
FROM x
WHERE prev_start IS NOT NULL;
""").df()


,median_step_minutes
0,[60]


* **변수 의미**

  * 반환 DF: `median_step_minutes` 제공함.
* **통계적 의미**

  * 윈도우 step이 실제로 60분인지 검증함. (실제로 60분이므로 문제 없음)

## 9. `ffill_gap_dist`: ffill이 끌어오는 “시간 갭” 분포(분위수)

* **하는 일**

  * ffill이 실제로 적용된 케이스만 대상으로,
  * `gap_minutes = window_start - prev_motor_time`를 계산함.
  * 분위수(10/25/50/75/90), 평균, 최소/최대를 요약함.

In [ ]:
ffill_gap_dist = con.execute("""
WITH in_window AS (
  SELECT
    w.stay_id,
    w.window_start,
    w.window_end,
    arg_max(g.gcs_motor, g.charttime) AS motor_last_in_window
  FROM windows_1h w
  LEFT JOIN gcs_events_all g
    ON g.stay_id = w.stay_id
   AND g.charttime >= w.window_start
   AND g.charttime <  w.window_end
  GROUP BY 1,2,3
),
ffill60 AS (
  SELECT
    iw.stay_id,
    iw.window_start,
    (
      SELECT g2.charttime
      FROM gcs_events_all g2
      WHERE g2.stay_id = iw.stay_id
        AND g2.gcs_motor IS NOT NULL
        AND g2.charttime < iw.window_start
        AND g2.charttime >= iw.window_start - INTERVAL '60 minutes'
      ORDER BY g2.charttime DESC
      LIMIT 1
    ) AS prev_motor_time
  FROM in_window iw
  WHERE iw.motor_last_in_window IS NULL
),
ffilled_only AS (
  SELECT
    stay_id,
    window_start,
    prev_motor_time,
    date_diff('minute', prev_motor_time, window_start) AS gap_minutes
  FROM ffill60
  WHERE prev_motor_time IS NOT NULL
)
SELECT
  COUNT(*) AS n_ffilled_windows,
  approx_quantile(gap_minutes, [0.1, 0.25, 0.5, 0.75, 0.9]) AS gap_quantiles_minutes,
  MIN(gap_minutes) AS min_gap,
  MAX(gap_minutes) AS max_gap,
  ROUND(AVG(gap_minutes), 2) AS mean_gap
FROM ffilled_only;
""").df()

ffill_gap_dist

,n_ffilled_windows,gap_quantiles_minutes,min_gap,max_gap,mean_gap
0,20159,"[5, 14, 29, 44, 53]",0,60,29.41


* **변수 의미**

  * `ffill_gap_dist`: ffill된 케이스의 gap 요약 DF임.
  * `gap_minutes`: 보정에 사용된 값이 얼마나 “과거”인지(분 단위) 나타냄.
* **통계적 의미**

  * gap이 작으면 “윈도우 경계 보정” 성격이 강함.
  * gap이 크면 “실제 forward fill(상태 복제)” 성격이 강해져 임상적/모델링 리스크가 커짐.
  * p50/p75/p90는 ffill cutoff(60분)가 과한지 판단하는 핵심 근거임.

## 10. `ffill_gap_bins`: gap 버킷 분포(≤5, 6–10, …, 31–60)

* **하는 일**

  * gap_minutes를 구간화하여 각 구간의 윈도우 수/비율을 계산함.

In [10]:
ffill_gap_bins = con.execute("""
WITH in_window AS (
  SELECT
    w.stay_id,
    w.window_start,
    arg_max(g.gcs_motor, g.charttime) AS motor_last_in_window
  FROM windows_1h w
  LEFT JOIN gcs_events_all g
    ON g.stay_id = w.stay_id
   AND g.charttime >= w.window_start
   AND g.charttime <  w.window_end
  GROUP BY 1,2
),
ffilled_only AS (
  SELECT
    iw.stay_id,
    date_diff(
      'minute',
      (
        SELECT g2.charttime
        FROM gcs_events_all g2
        WHERE g2.stay_id = iw.stay_id
          AND g2.gcs_motor IS NOT NULL
          AND g2.charttime < iw.window_start
          AND g2.charttime >= iw.window_start - INTERVAL '60 minutes'
        ORDER BY g2.charttime DESC
        LIMIT 1
      ),
      iw.window_start
    ) AS gap_minutes
  FROM in_window iw
  WHERE iw.motor_last_in_window IS NULL
)
SELECT
  CASE
    WHEN gap_minutes <= 5  THEN '≤5 min'
    WHEN gap_minutes <= 10 THEN '6–10 min'
    WHEN gap_minutes <= 20 THEN '11–20 min'
    WHEN gap_minutes <= 30 THEN '21–30 min'
    WHEN gap_minutes <= 60 THEN '31–60 min'
  END AS gap_bucket,
  COUNT(*) AS n_windows,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM ffilled_only
GROUP BY 1
ORDER BY
  CASE gap_bucket
    WHEN '≤5 min' THEN 1
    WHEN '6–10 min' THEN 2
    WHEN '11–20 min' THEN 3
    WHEN '21–30 min' THEN 4
    WHEN '31–60 min' THEN 5
  END;
""").df()

ffill_gap_bins


,gap_bucket,n_windows,pct
0,≤5 min,2040,2.46
1,6–10 min,1638,1.98
2,11–20 min,3360,4.05
3,21–30 min,3461,4.17
4,31–60 min,9660,11.65
5,None,62748,75.68


* **변수 의미**

  * `ffill_gap_bins`: 구간별 분포 DF임.
* **통계적 의미**

  * ffill이 “대부분 5분 이내”인지, “30–60분이 많이 섞이는지”를 직관적으로 보여줌.
  * cutoff를 60분 유지할지/30분으로 줄일지의 의사결정 근거로 사용함.

## 11. `gcs_features_1h` 뷰 생성(최종 피처 설계 반영)

* **하는 일**

  * 각 1시간 윈도우에서 E/V/M 각각에 대해 다음을 생성함.

    * `gcs_*_first`: 윈도우 내 최초값(`arg_min`)
    * `gcs_*_latest`: 윈도우 내 최신값(`arg_max`)
    * `gcs_*_trend`: latest - first(둘 다 존재할 때만)
    * `gcs_*_latest_prev60`: 직전 60분 내 최신값(보정 후보)
    * `gcs_*_latest_ffill60`: latest가 없으면 prev60으로 대체
    * `gcs_*_used_ffill60_flag`: latest가 NULL이고 prev60이 있으면 1

In [11]:
con.execute("""
CREATE OR REPLACE TEMP VIEW gcs_features_1h AS
WITH in_window AS (
  SELECT
    w.stay_id,
    w.window_start,
    w.window_end,

    -- Motor: first / latest
    arg_min(g.gcs_motor, g.charttime) AS gcs_motor_first,
    arg_max(g.gcs_motor, g.charttime) AS gcs_motor_latest,

    -- Verbal: first / latest
    arg_min(g.gcs_verbal, g.charttime) AS gcs_verbal_first,
    arg_max(g.gcs_verbal, g.charttime) AS gcs_verbal_latest,

    -- Eye: first / latest
    arg_min(g.gcs_eye, g.charttime) AS gcs_eye_first,
    arg_max(g.gcs_eye, g.charttime) AS gcs_eye_latest

  FROM windows_1h w
  LEFT JOIN gcs_events_all g
    ON g.stay_id = w.stay_id
   AND g.charttime >= w.window_start
   AND g.charttime <  w.window_end
  GROUP BY 1,2,3
),
ffill60 AS (
  SELECT
    iw.*,

    -- latest가 없을 때만 쓸 후보: 직전 60분 내 마지막 값 (component별)
    (
      SELECT g2.gcs_motor
      FROM gcs_events_all g2
      WHERE g2.stay_id = iw.stay_id
        AND g2.gcs_motor IS NOT NULL
        AND g2.charttime < iw.window_start
        AND g2.charttime >= iw.window_start - INTERVAL '60 minutes'
      ORDER BY g2.charttime DESC
      LIMIT 1
    ) AS gcs_motor_latest_prev60,

    (
      SELECT g2.gcs_verbal
      FROM gcs_events_all g2
      WHERE g2.stay_id = iw.stay_id
        AND g2.gcs_verbal IS NOT NULL
        AND g2.charttime < iw.window_start
        AND g2.charttime >= iw.window_start - INTERVAL '60 minutes'
      ORDER BY g2.charttime DESC
      LIMIT 1
    ) AS gcs_verbal_latest_prev60,

    (
      SELECT g2.gcs_eye
      FROM gcs_events_all g2
      WHERE g2.stay_id = iw.stay_id
        AND g2.gcs_eye IS NOT NULL
        AND g2.charttime < iw.window_start
        AND g2.charttime >= iw.window_start - INTERVAL '60 minutes'
      ORDER BY g2.charttime DESC
      LIMIT 1
    ) AS gcs_eye_latest_prev60

  FROM in_window iw
),
final AS (
  SELECT
    stay_id,
    window_start,
    window_end,

    -- ===== Motor =====
    gcs_motor_first,
    gcs_motor_latest,
    CASE
      WHEN gcs_motor_first IS NOT NULL AND gcs_motor_latest IS NOT NULL
        THEN gcs_motor_latest - gcs_motor_first
      ELSE NULL
    END AS gcs_motor_trend,

    -- "원본 latest"가 비면 ffill60 참고(하지만 원본도 함께 보존)
    COALESCE(gcs_motor_latest, gcs_motor_latest_prev60) AS gcs_motor_latest_ffill60,
    CASE
      WHEN gcs_motor_latest IS NULL AND gcs_motor_latest_prev60 IS NOT NULL THEN 1
      ELSE 0
    END AS gcs_motor_used_ffill60_flag,

    -- ===== Verbal =====
    gcs_verbal_first,
    gcs_verbal_latest,
    CASE
      WHEN gcs_verbal_first IS NOT NULL AND gcs_verbal_latest IS NOT NULL
        THEN gcs_verbal_latest - gcs_verbal_first
      ELSE NULL
    END AS gcs_verbal_trend,

    COALESCE(gcs_verbal_latest, gcs_verbal_latest_prev60) AS gcs_verbal_latest_ffill60,
    CASE
      WHEN gcs_verbal_latest IS NULL AND gcs_verbal_latest_prev60 IS NOT NULL THEN 1
      ELSE 0
    END AS gcs_verbal_used_ffill60_flag,

    -- ===== Eye =====
    gcs_eye_first,
    gcs_eye_latest,
    CASE
      WHEN gcs_eye_first IS NOT NULL AND gcs_eye_latest IS NOT NULL
        THEN gcs_eye_latest - gcs_eye_first
      ELSE NULL
    END AS gcs_eye_trend,

    COALESCE(gcs_eye_latest, gcs_eye_latest_prev60) AS gcs_eye_latest_ffill60,
    CASE
      WHEN gcs_eye_latest IS NULL AND gcs_eye_latest_prev60 IS NOT NULL THEN 1
      ELSE 0
    END AS gcs_eye_used_ffill60_flag

  FROM ffill60
)
SELECT * FROM final;
""")



* **변수 의미**

  * `gcs_features_1h`: (stay_id, window_start, window_end) 기준으로 GCS 피처가 붙은 뷰임.
* **통계적 의미**

  * `first/latest/trend`는 “윈도우 내부 변화(동적 정보)”를 요약하는 통계량임.
  * `latest_ffill60`은 결측을 줄이기 위한 “보조값”임.
  * `used_ffill60_flag`는 값의 생성 메커니즘을 노출함(관측 vs 보정의 식별자) → 모델이 측정 공백 자체도 학습 가능하게 함.

## 12. `gcs_features_1h` 생성 결과 점검

* **하는 일**: 스키마(`DESCRIBE`)와 샘플 행(`LIMIT 5`)을 확인함.

In [12]:
con.execute("DESCRIBE gcs_features_1h").df()
con.execute("SELECT * FROM gcs_features_1h LIMIT 5").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,stay_id,window_start,window_end,gcs_motor_first,gcs_motor_latest,gcs_motor_trend,gcs_motor_latest_ffill60,gcs_motor_used_ffill60_flag,gcs_verbal_first,gcs_verbal_latest,gcs_verbal_trend,gcs_verbal_latest_ffill60,gcs_verbal_used_ffill60_flag,gcs_eye_first,gcs_eye_latest,gcs_eye_trend,gcs_eye_latest_ffill60,gcs_eye_used_ffill60_flag
0,36849644,2144-05-03 13:24:53,2144-05-03 19:24:53,6.0,6.0,0.0,6.0,0,4.0,4.0,0.0,4.0,0,3.0,3.0,0.0,3.0,0
1,35574490,2155-03-25 03:08:13,2155-03-25 09:08:13,6.0,6.0,0.0,6.0,0,5.0,5.0,0.0,5.0,0,3.0,4.0,1.0,4.0,0
2,33529344,2173-09-13 07:30:17,2173-09-13 13:30:17,6.0,6.0,0.0,6.0,0,4.0,4.0,0.0,4.0,0,3.0,3.0,0.0,3.0,0
3,34541647,2125-07-16 12:22:00,2125-07-16 18:22:00,6.0,6.0,0.0,6.0,0,4.0,4.0,0.0,4.0,0,4.0,4.0,0.0,4.0,0
4,34541647,2125-07-16 14:22:00,2125-07-16 20:22:00,6.0,6.0,0.0,6.0,0,4.0,4.0,0.0,4.0,0,4.0,4.0,0.0,4.0,0


* **통계적 의미**

  * 피처가 의도대로 생성되었는지(타입/NULL/컬럼 존재) 검증함.
  * 특히 timestamp 키 정합성(조인 실패 방지) 확인용임.

## 13. `windows_raw` 뷰 생성(원본 CSV 전체 컬럼 유지)

* **하는 일**

  * CSV 전체를 `windows_raw`로 로드함(원본 컬럼 보존 목적).

In [13]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW windows_raw AS
SELECT *
FROM read_csv_auto('{window_csv_path}') AS t;
""")

* **변수 의미**

  * `windows_raw`: sliding window CSV 원본 전체 컬럼을 가진 뷰임.
* **통계적 의미**

  * 이후 최종 테이블에서 label(예: mortality/vent 등)과 기존 feature들을 유지한 채 GCS 피처를 결합하기 위한 기반임.

## 14. `final_1h_with_gcs` 생성 및 확인(full_table)

* **하는 일**

  * `windows_raw`에 `gcs_features_1h`를 (stay_id + start/end time)로 LEFT JOIN함.
  * 조인 결과를 뷰로 만들고 DF로 출력함.

In [14]:
full_table = con.execute("""
CREATE OR REPLACE TEMP VIEW final_1h_with_gcs AS
SELECT
  w.*,
  g.*
FROM windows_raw w
LEFT JOIN gcs_features_1h g
  ON g.stay_id = w.stay_id
 AND g.window_start = w.observation_start_time
 AND g.window_end   = w.observation_end_time;
""").df()

full_table

,Count


* **변수 의미**

  * `full_table`: 조인 결과 DF임(뷰 생성 쿼리를 실행하며 반환).
  * `final_1h_with_gcs`: 최종 분석 테이블(원본 window + GCS 피처 결합) 뷰임.
* **통계적 의미**

  * LEFT JOIN 사용으로 윈도우 전체를 분모로 유지함(결측이 생겨도 윈도우가 탈락하지 않음).
  * 이 테이블이 실제 모델 입력 프레임이 됨.

---

## 15. `ffill_summary`: component별 ffill 사용률 요약

* **하는 일**

  * Motor/Verbal/Eye 각각에 대해

    * 원본 latest 존재 수
    * ffill60 후 존재 수
    * ffill 실제 사용 수(used flag=1)
    * ffill 사용률(%)
    
    을 계산함.

In [15]:
ffill_summary = con.execute("""
SELECT
  COUNT(*) AS n_rows,

  -- Motor
  SUM(CASE WHEN gcs_motor_latest IS NOT NULL THEN 1 ELSE 0 END) AS motor_latest_nonnull,
  SUM(CASE WHEN gcs_motor_latest_ffill60 IS NOT NULL THEN 1 ELSE 0 END) AS motor_latest_ffill60_nonnull,
  SUM(CASE WHEN gcs_motor_used_ffill60_flag = 1 THEN 1 ELSE 0 END) AS motor_ffill_used_rows,
  ROUND(100.0 * SUM(CASE WHEN gcs_motor_used_ffill60_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS motor_ffill_used_pct,

  -- Verbal
  SUM(CASE WHEN gcs_verbal_latest IS NOT NULL THEN 1 ELSE 0 END) AS verbal_latest_nonnull,
  SUM(CASE WHEN gcs_verbal_latest_ffill60 IS NOT NULL THEN 1 ELSE 0 END) AS verbal_latest_ffill60_nonnull,
  SUM(CASE WHEN gcs_verbal_used_ffill60_flag = 1 THEN 1 ELSE 0 END) AS verbal_ffill_used_rows,
  ROUND(100.0 * SUM(CASE WHEN gcs_verbal_used_ffill60_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS verbal_ffill_used_pct,

  -- Eye
  SUM(CASE WHEN gcs_eye_latest IS NOT NULL THEN 1 ELSE 0 END) AS eye_latest_nonnull,
  SUM(CASE WHEN gcs_eye_latest_ffill60 IS NOT NULL THEN 1 ELSE 0 END) AS eye_latest_ffill60_nonnull,
  SUM(CASE WHEN gcs_eye_used_ffill60_flag = 1 THEN 1 ELSE 0 END) AS eye_ffill_used_rows,
  ROUND(100.0 * SUM(CASE WHEN gcs_eye_used_ffill60_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS eye_ffill_used_pct

FROM final_1h_with_gcs;
""").df()

ffill_summary


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,motor_latest_nonnull,motor_latest_ffill60_nonnull,motor_ffill_used_rows,motor_ffill_used_pct,verbal_latest_nonnull,verbal_latest_ffill60_nonnull,verbal_ffill_used_rows,verbal_ffill_used_pct,eye_latest_nonnull,eye_latest_ffill60_nonnull,eye_ffill_used_rows,eye_ffill_used_pct
0,934312,851405.0,871564.0,20159.0,2.16,853165.0,873162.0,19997.0,2.14,853373.0,873314.0,19941.0,2.13


* **변수 의미**

  * `ffill_summary`: component별 요약 DF임.
* **통계적 의미**

  * component별 ffill 사용률이 비슷하면 “특정 컴포넌트만 편향적으로 보정되는 문제”가 적다고 해석 가능함.
  * 결측의 원인이 “component 특성”보다 “측정/기록 공백 구간”일 가능성을 시사함.

## 16. `all_empty_summary`: 세 컴포넌트가 모두 비는 구간(all_three_empty) 분석

* **하는 일**

  * `motor_all_empty / verbal_all_empty / eye_all_empty`를 first/latest 동시 NULL 여부로 정의함.
  * 세 개 모두 empty인 `all_three_empty`를 계산함.
  * all_three_empty 비율(%) 및 그 중 motor ffill60으로라도 구제되는 비율(%)을 계산함.


In [16]:
all_empty_summary = con.execute("""
WITH x AS (
  SELECT
    *,
    CASE WHEN gcs_motor_first IS NULL AND gcs_motor_latest IS NULL THEN 1 ELSE 0 END AS motor_all_empty,
    CASE WHEN gcs_verbal_first IS NULL AND gcs_verbal_latest IS NULL THEN 1 ELSE 0 END AS verbal_all_empty,
    CASE WHEN gcs_eye_first IS NULL AND gcs_eye_latest IS NULL THEN 1 ELSE 0 END AS eye_all_empty
  FROM final_1h_with_gcs
),
y AS (
  SELECT
    *,
    CASE WHEN motor_all_empty=1 AND verbal_all_empty=1 AND eye_all_empty=1 THEN 1 ELSE 0 END AS all_three_empty
  FROM x
)
SELECT
  COUNT(*) AS n_rows,
  SUM(all_three_empty) AS n_all_three_empty,
  ROUND(100.0 * SUM(all_three_empty) / COUNT(*), 2) AS pct_all_three_empty,

  -- all_three_empty인데도 ffill로라도 motor_latest를 얻은 경우
  SUM(CASE WHEN all_three_empty=1 AND gcs_motor_latest_ffill60 IS NOT NULL THEN 1 ELSE 0 END) AS n_all_three_empty_but_motor_ffilled,
  ROUND(
    100.0 * SUM(CASE WHEN all_three_empty=1 AND gcs_motor_latest_ffill60 IS NOT NULL THEN 1 ELSE 0 END)
    / NULLIF(SUM(all_three_empty), 0),
    2
  ) AS pct_rescued_by_motor_ffill_within_all_three_empty

FROM y;
""").df()

all_empty_summary


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_all_three_empty,pct_all_three_empty,n_all_three_empty_but_motor_ffilled,pct_rescued_by_motor_ffill_within_all_three_empty
0,934312,79170.0,8.47,19575.0,24.73


* **변수 의미**

  * `all_empty_summary`: 완전 공백 구간 규모 및 구제율 DF임.
* **통계적 의미**

  * 결측의 핵심이 “부분 결측”인지 “완전 공백”인지 판단하는 지표임.
  * `pct_rescued_by_motor_ffill_within_all_three_empty`는 ffill이 “완전 공백”을 얼마나 줄이는지(안전망 효과)를 정량화함.

## 17. `flag_pattern`: ffill flag 조합 패턴(0/1) 분포

* **하는 일**

  * (motor_ffill, verbal_ffill, eye_ffill) 조합별 윈도우 수/비율을 집계함.

In [17]:
flag_pattern = con.execute("""
SELECT
  gcs_motor_used_ffill60_flag AS motor_ffill,
  gcs_verbal_used_ffill60_flag AS verbal_ffill,
  gcs_eye_used_ffill60_flag AS eye_ffill,
  COUNT(*) AS n_rows,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM final_1h_with_gcs
GROUP BY 1,2,3
ORDER BY n_rows DESC;
""").df()

flag_pattern


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,motor_ffill,verbal_ffill,eye_ffill,n_rows,pct
0,0,0,0,913637,97.79
1,1,1,1,19469,2.08
2,1,0,0,338,0.04
3,0,1,0,213,0.02
4,1,1,0,183,0.02
5,0,0,1,171,0.02
6,1,0,1,169,0.02
7,0,1,1,132,0.01


* **변수 의미**

  * `flag_pattern`: 3개 flag의 조합 분포 DF임.
* **통계적 의미**

  * (1,1,1)이 대부분이면 ffill이 “세 컴포넌트가 함께 비는 공백 구간”에서 주로 발생한다고 해석 가능함.
  * 일부만 1인 조합이 많으면 component 비동기 보정이 많아져 피처 일관성 문제가 커질 수 있음.

## 18. csv 형태로 저장

In [ ]:
gcs_out_csv = "../data/processed/gcs_features_1h.csv"

con.execute(f"""
COPY (
  SELECT * FROM gcs_features_1h
) TO '{gcs_out_csv}' (HEADER, DELIMITER ',');
""")

print("Saved:", gcs_out_csv)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: ../data/processed/gcs_features_1h.csv


## 19. sliding.csv + gcs_features_1h.csv 조인해서 확인 + 최종 CSV 저장

In [ ]:
import pandas as pd

sliding_csv_path = "../data/processed/cohort_sliding_window_v2.csv"   # 원본 sliding
gcs_out_csv      = "../data/processed/gcs_features_1h.csv"           # 위에서 만든 gcs 피처

sliding = pd.read_csv(sliding_csv_path, parse_dates=["observation_start_time","observation_end_time"])
gcs     = pd.read_csv(gcs_out_csv,      parse_dates=["window_start","window_end"])

merged = sliding.merge(
    gcs,
    left_on=["stay_id","observation_start_time","observation_end_time"],
    right_on=["stay_id","window_start","window_end"],
    how="left"
)

merged.head()


,subject_id,hadm_id,stay_id,intime,outtime,los,first_careunit,last_careunit,anchor_age,gender,...,gcs_verbal_first,gcs_verbal_latest,gcs_verbal_trend,gcs_verbal_latest_ffill60,gcs_verbal_used_ffill60_flag,gcs_eye_first,gcs_eye_latest,gcs_eye_trend,gcs_eye_latest_ffill60,gcs_eye_used_ffill60_flag
0,19958279,27775101,37262027,2177-12-05 04:15:00,2177-12-07 21:33:40,2.721296,Neuro Stepdown,Neuro Stepdown,73,F,...,5.0,5.0,0.0,5.0,0,3.0,3.0,0.0,3.0,0
1,19967825,21175076,33653109,2187-07-27 23:47:19,2187-07-30 15:29:14,2.654109,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),91,F,...,5.0,5.0,0.0,5.0,0,4.0,4.0,0.0,4.0,0
2,19985259,23988340,38791957,2129-12-19 20:48:00,2129-12-22 13:45:59,2.706933,Coronary Care Unit (CCU),Coronary Care Unit (CCU),70,M,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
3,19996432,29434165,36469520,2111-11-13 09:06:00,2111-11-14 18:44:57,1.402049,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),60,M,...,5.0,5.0,0.0,5.0,0,3.0,4.0,1.0,4.0,0
4,18593026,24839346,36149581,2174-05-18 15:02:00,2174-05-19 16:54:29,1.078113,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),27,F,...,5.0,5.0,0.0,5.0,0,4.0,4.0,0.0,4.0,0


In [23]:
n = len(merged)
motor_latest_nonnull = merged["gcs_motor_latest"].notna().sum()
motor_ffill_nonnull  = merged["gcs_motor_latest_ffill60"].notna().sum()
motor_ffill_used     = merged["gcs_motor_used_ffill60_flag"].fillna(0).astype(int).sum()

print("rows:", n)
print("motor_latest_nonnull:", motor_latest_nonnull, motor_latest_nonnull/n)
print("motor_ffill60_nonnull:", motor_ffill_nonnull, motor_ffill_nonnull/n)
print("motor_ffill_used:", motor_ffill_used, motor_ffill_used/n)


rows: 934312
motor_latest_nonnull: 851405 0.9112641173398179
motor_ffill60_nonnull: 871564 0.9328404216150493
motor_ffill_used: 20159 0.0215763042752314


In [25]:
out_merged_csv = "../data/processed/cohort_sliding_window_v2_with_gcs.csv"
merged.to_csv(out_merged_csv, index=False)
print("Saved:", out_merged_csv)

Saved: ../data/processed/cohort_sliding_window_v2_with_gcs.csv


## 20. 최종 csv 파일 재확인

### 1) CSV row/column/헤더 정상 여부 빠르게 확인 (pandas)

In [28]:
import pandas as pd

csv_path = "../data/processed/cohort_sliding_window_v2_with_gcs.csv"

df = pd.read_csv(csv_path)

print("shape:", df.shape)      # (rows, cols)
print("columns:", len(df.columns))
print("head:")
display(df.head(3))
print("tail:")
display(df.tail(3))


shape: (934312, 53)
columns: 53
head:


,subject_id,hadm_id,stay_id,intime,outtime,los,first_careunit,last_careunit,anchor_age,gender,...,gcs_verbal_first,gcs_verbal_latest,gcs_verbal_trend,gcs_verbal_latest_ffill60,gcs_verbal_used_ffill60_flag,gcs_eye_first,gcs_eye_latest,gcs_eye_trend,gcs_eye_latest_ffill60,gcs_eye_used_ffill60_flag
0,19958279,27775101,37262027,2177-12-05 04:15:00,2177-12-07 21:33:40,2.721296,Neuro Stepdown,Neuro Stepdown,73,F,...,5.0,5.0,0.0,5.0,0,3.0,3.0,0.0,3.0,0
1,19967825,21175076,33653109,2187-07-27 23:47:19,2187-07-30 15:29:14,2.654109,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),91,F,...,5.0,5.0,0.0,5.0,0,4.0,4.0,0.0,4.0,0
2,19985259,23988340,38791957,2129-12-19 20:48:00,2129-12-22 13:45:59,2.706933,Coronary Care Unit (CCU),Coronary Care Unit (CCU),70,M,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0


tail:


,subject_id,hadm_id,stay_id,intime,outtime,los,first_careunit,last_careunit,anchor_age,gender,...,gcs_verbal_first,gcs_verbal_latest,gcs_verbal_trend,gcs_verbal_latest_ffill60,gcs_verbal_used_ffill60_flag,gcs_eye_first,gcs_eye_latest,gcs_eye_trend,gcs_eye_latest_ffill60,gcs_eye_used_ffill60_flag
934309,16861883,27416514,34135517,2116-05-06 01:48:00,2116-05-11 16:56:51,5.631146,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),87,M,...,5.0,5.0,0.0,5.0,0,4.0,4.0,0.0,4.0,0
934310,17001770,27416903,39593478,2133-08-28 15:17:50,2133-08-29 15:44:25,1.018461,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),53,M,...,5.0,5.0,0.0,5.0,0,4.0,4.0,0.0,4.0,0
934311,17313458,28194280,33543545,2121-12-09 20:56:16,2121-12-11 15:39:02,1.779699,Trauma SICU (TSICU),Trauma SICU (TSICU),91,F,...,4.0,4.0,0.0,4.0,0,4.0,3.0,-1.0,3.0,0


### 2) 조인이 실제로 붙었는지 확인
#### (A) gcs 컬럼이 존재하는지

In [29]:
gcs_cols = [c for c in df.columns if c.startswith("gcs_")]
print("n_gcs_cols:", len(gcs_cols))
print(gcs_cols)


n_gcs_cols: 15
['gcs_motor_first', 'gcs_motor_latest', 'gcs_motor_trend', 'gcs_motor_latest_ffill60', 'gcs_motor_used_ffill60_flag', 'gcs_verbal_first', 'gcs_verbal_latest', 'gcs_verbal_trend', 'gcs_verbal_latest_ffill60', 'gcs_verbal_used_ffill60_flag', 'gcs_eye_first', 'gcs_eye_latest', 'gcs_eye_trend', 'gcs_eye_latest_ffill60', 'gcs_eye_used_ffill60_flag']


#### (B) motor_latest / motor_latest_ffill60 / flag 값이 실제로 들어갔는지

In [30]:
checks = {
    "motor_latest_notnull": df["gcs_motor_latest"].notna().mean(),
    "motor_ffill60_notnull": df["gcs_motor_latest_ffill60"].notna().mean(),
    "motor_ffill_used_rate": df["gcs_motor_used_ffill60_flag"].fillna(0).astype(int).mean(),
}
checks

{'motor_latest_notnull': 0.9112641173398179,
 'motor_ffill60_notnull': 0.9328404216150493,
 'motor_ffill_used_rate': 0.0215763042752314}

### 3) sliding 원본과 행이 1:1로 유지됐는지

In [ ]:
sliding_path = "../data/processed/cohort_sliding_window_v2.csv"

sl = pd.read_csv(sliding_path)
print("sliding rows:", len(sl))
print("merged  rows:", len(df))
print("same rows:", len(sl) == len(df))

sliding rows: 934312
merged  rows: 934312
same rows: True


### 4) 조인 키 중복/불일치 검사
#### (A) merged에서 조인 키 중복 확인

In [32]:
key = ["stay_id", "observation_start_time", "observation_end_time"]
dup = df.duplicated(key).sum()
print("duplicate keys in merged:", dup)


duplicate keys in merged: 0


#### (B) gcs 값이 너무 많이 NULL이면 타임스탬프 타입 불일치

In [33]:
null_rate = df["gcs_motor_latest_ffill60"].isna().mean()
print("motor_latest_ffill60 null rate:", null_rate)


motor_latest_ffill60 null rate: 0.06715957838495064


### 5) 표본으로 “진짜로 붙었는지” 5건만 눈으로 확인

In [34]:
cols_to_view = [
    "stay_id", "observation_start_time", "observation_end_time",
    "gcs_motor_first", "gcs_motor_latest", "gcs_motor_trend",
    "gcs_motor_latest_ffill60", "gcs_motor_used_ffill60_flag"
]
display(df[cols_to_view].sample(5, random_state=42))


,stay_id,observation_start_time,observation_end_time,gcs_motor_first,gcs_motor_latest,gcs_motor_trend,gcs_motor_latest_ffill60,gcs_motor_used_ffill60_flag
617521,38368764,2141-01-12 08:18:00,2141-01-12 14:18:00,6.0,6.0,0.0,6.0,0
506468,35676485,2147-08-31 20:21:03,2147-09-01 02:21:03,6.0,6.0,0.0,6.0,0
46989,39506882,2171-06-19 17:41:04,2171-06-19 23:41:04,1.0,1.0,0.0,1.0,0
313084,39867844,2131-08-16 14:01:00,2131-08-16 20:01:00,6.0,6.0,0.0,6.0,0
855366,33840003,2118-08-17 18:23:04,2118-08-18 00:23:04,6.0,6.0,0.0,6.0,0
